# Ajuste de YOLO para Deteccion de Eventos Sonoros (YOHO)

## Por que YOLO para audio

Nuestro proyecto reformula la Deteccion de Eventos Sonoros (SED) como un problema de deteccion de objetos en el plano tiempo-frecuencia. En lugar de tratar el audio como una secuencia temporal pura, lo convertimos en una imagen (espectrograma) donde cada evento sonoro es un objeto que ocupa una region especifica.

Esta aproximacion esta respaldada por literatura reciente: trabajos como el de Gonzalez et al. (2026) demuestran que adaptar detectores modernos como YOLOv8 a espectrogramas mel es competitivo frente a arquitecturas tradicionales como CRNN o Conformer, con ventajas en condiciones de solapamiento y ruido. Ademas, el propio nombre YOHO (You Only Hear Once) es una reinterpretacion deliberada del paradigma single-pass de YOLO: procesamos la representacion visual del audio en una sola pasada para detectar eventos.

## Que necesitamos ajustar

YOLO esta disenado para imagenes RGB de 3 canales, pero nuestro dataset usa espectrogramas en escala de grises de 1 canal (PNG de 16 bits, modo I;16). Necesitamos indicarle a YOLO que trabaje con `channels: 1`. Ultralytics anadio soporte oficial para esto en versiones recientes (v8.3.131+), con un parametro `channels` en el data loader y la posibilidad de declararlo directamente en el `data.yaml`. La documentacion oficial confirma que basta con anadir `channels: 1` al YAML para que YOLO convierta las imagenes a escala de grises durante el entrenamiento.

## Que configuracion usaremos

| Parametro | Valor | Justificacion |
|---|---|---|
| Modelo | `yolo11n.pt` (o `yolov8n.pt`) | Modelo nano, rapido, adecuado para validar el pipeline |
| `imgsz` | 1024 | Nuestras imagenes miden aproximadamente 1032x1024 px; 1024 es multiplo de 32 y el mas cercano |
| `epochs` | 100 | Estandar; con `patience` se detiene antes si no mejora |
| `batch` | -1 (AutoBatch) | Ultralytics ajusta automaticamente el maximo que cabe en 8 GB VRAM |
| `patience` | 30 | Early stopping para ahorrar tiempo |
| `device` | 0 | Usa la RTX 5060 |
| `channels` | 1 | Ya esta en el `data.yaml` generado |

In [ ]:
import sys
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from PIL import Image
from collections import Counter

sys.path.append("../src")
from config import (
    DATASET_ROOT,
    AUDIO_DIR,
    CSV_MONO,
    OUTPUT_ROOT,
    SPECTROGRAM_DIR,
    LABELS_DIR,
)

In [ ]:
# Verificar que el data.yaml existe y mostrar su contenido
data_yaml_path = OUTPUT_ROOT / "data.yaml"
print(f"Ruta del data.yaml: {data_yaml_path}")
print(f"Existe: {data_yaml_path.exists()}")
print("\n--- Contenido del data.yaml ---")
with open(data_yaml_path, 'r') as f:
    print(f.read())

## Sobre el `channels: 1` y los pesos preentrenados

Cuando usamos `channels: 1`, Ultralytics reinicia los pesos de la primera capa convolucional (la que recibe la imagen), porque los pesos originales de `yolo11n.pt` esperan 3 canales RGB. El resto del modelo (backbone, neck, head) conserva los pesos preentrenados en COCO. Esto es normal y esperado: la red aprendera a extraer caracteristicas de nuestros espectrogramas en esa primera capa durante el entrenamiento.

En el informe podemos mencionarlo como una decision de diseno: "Se utilizo el soporte nativo de Ultralytics para entrada de 1 canal (`channels: 1`), lo que implica que la primera capa convolucional se inicializa aleatoriamente mientras el resto del modelo conserva los pesos preentrenados en COCO."

In [ ]:
splits = ['train', 'val', 'test']
for split in splits:
    img_dir = OUTPUT_ROOT / split
    if not img_dir.exists():
        print(f"{split}: carpeta no encontrada")
        continue
    imgs = list(img_dir.glob("*.png"))
    if not imgs:
        print(f"{split}: sin imagenes")
        continue
    modos = Counter(Image.open(p).mode for p in imgs[:20])
    print(f"{split}: {len(imgs)} imagenes | Modos (primeras 20): {dict(modos)}")

In [ ]:
train_script = '''
from ultralytics import YOLO
from pathlib import Path
import sys

sys.path.append(str(Path(__file__).resolve().parent))
from config import OUTPUT_ROOT

def main():
    data_yaml = OUTPUT_ROOT / "data.yaml"
    model = YOLO("yolo11n.pt")

    results = model.train(
        data=str(data_yaml),
        epochs=100,
        imgsz=1024,
        batch=-1,
        patience=30,
        workers=4,
        device=0,
        project=str(OUTPUT_ROOT / "runs"),
        name="yoho_train",
        exist_ok=True,
        pretrained=True,
        verbose=True,
    )
    print("Entrenamiento finalizado.")
    print(f"Resultados en: {results.save_dir}")

if __name__ == "__main__":
    main()
'''

train_path = Path("../src/train.py")
train_path.write_text(train_script)
print(f"Script guardado en: {train_path}")
print("\n--- Contenido de train.py ---")
print(train_script)

## Consideraciones antes de lanzar el entrenamiento

1. Tamano del dataset: aproximadamente 3,541 ventanas (tras el split, aproximadamente 2,833 train, 354 val, 354 test). Es un tamano manejable para una GPU de 8 GB.
2. Ventanas sin cajas: El script de generacion ya creo archivos `.txt` vacios para ventanas sin eventos. YOLO las tratara como ejemplos negativos (fondo), lo que ayuda a reducir falsos positivos.
3. Metricas a revisar: Tras el entrenamiento, revisaremos `mAP@0.5`, `mAP@0.5:0.95`, la matriz de confusion y las curvas Precision-Recall. Prestaremos especial atencion a las clases con menos muestras (posible desbalance) y a las de eventos cortos.
4. Posibles ajustes: Si el entrenamiento se estanca o hay sobreajuste, podremos ajustar `patience`, `batch`, o probar data augmentation especifica para audio (ruido, desplazamiento temporal).

## Siguiente paso

Ejecutar `python src/train.py` desde la terminal y monitorear el progreso. Una vez finalizado, analizaremos las metricas en el notebook `03_resultados.ipynb`.